In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mysql.connector

db = mysql.connector.connect(host = "localhost",
                             username = "root",
                             password = "root1234@#",
                             database = "ecommerce")


                        

# cal moving avg

In [11]:
cur = db.cursor()

query = """select customer_id, order_purchase_timestamp,payment,
avg(payment) over (partition by customer_id order by order_purchase_timestamp
rows between 2 preceding and  current row) as mov_avg
from
(select orders.customer_id,orders.order_purchase_timestamp,
payments.payment_value as payment
from payments join orders
on payments.order_id = orders.order_id) as a"""

cur.execute(query)
data = cur.fetchall()
data

df = pd.DataFrame(data)
df



,0,1,2,3
0,00012a2ce6f8dcda20d059ce98491703,2017-11-14 16:08:26,114.74,114.739998
1,00012a2ce6f8dcda20d059ce98491703,2017-11-14 16:08:26,114.74,114.739998
2,00012a2ce6f8dcda20d059ce98491703,2017-11-14 16:08:26,114.74,114.739998
3,00012a2ce6f8dcda20d059ce98491703,2017-11-14 16:08:26,114.74,114.739998
4,000161a058600d5901f007fab4c27140,2017-07-16 09:40:32,67.41,67.410004
...,...,...,...,...
415539,ffffa3172527f765de70084a7e53aae8,2017-09-02 11:53:32,45.50,45.500000
415540,ffffe8b65bbe3087b653a978c870db99,2017-09-29 14:07:03,18.37,18.370001
415541,ffffe8b65bbe3087b653a978c870db99,2017-09-29 14:07:03,18.37,18.370001
415542,ffffe8b65bbe3087b653a978c870db99,2017-09-29 14:07:03,18.37,18.370001


# cum sales per month per year

In [12]:
query = """SELECT 
    years,
    months,
    payments,
    SUM(payments) OVER(ORDER BY years, months) AS cum_sales
FROM
(
    SELECT 
        YEAR(orders.order_purchase_timestamp) AS years,
        MONTH(orders.order_purchase_timestamp) AS months,
        ROUND(SUM(payments.payment_value),2) AS payments
    FROM orders
    JOIN payments
        ON orders.order_id = payments.order_id
    GROUP BY years, months
    ORDER BY years, months
) AS a """

cur.execute(query)
data = cur.fetchall()
data


[(2016, 9, 1008.96, 1008.96),
 (2016, 10, 236361.92, 237370.88),
 (2016, 12, 78.48, 237449.36000000002),
 (2017, 1, 553952.16, 791401.52),
 (2017, 2, 1167632.04, 1959033.56),
 (2017, 3, 1799454.4, 3758487.96),
 (2017, 4, 1671152.12, 5429640.08),
 (2017, 5, 2371675.28, 7801315.359999999),
 (2017, 6, 2045105.52, 9846420.879999999),
 (2017, 7, 2369531.68, 12215952.559999999),
 (2017, 8, 2697585.28, 14913537.839999998),
 (2017, 9, 2911049.8, 17824587.639999997),
 (2017, 10, 3118711.52, 20943299.159999996),
 (2017, 11, 4779531.2, 25722830.359999996),
 (2017, 12, 3513605.92, 29236436.279999994),
 (2018, 1, 4460016.72, 33696452.99999999),
 (2018, 2, 3969853.36, 37666306.35999999),
 (2018, 3, 4638608.48, 42304914.83999999),
 (2018, 4, 4643141.92, 46948056.75999999),
 (2018, 5, 4615928.6, 51563985.35999999),
 (2018, 6, 4095522.0, 55659507.35999999),
 (2018, 7, 4266163.0, 59925670.35999999),
 (2018, 8, 4089701.29, 64015371.64999999),
 (2018, 9, 17758.16, 64033129.80999999),
 (2018, 10, 2358.68, 

# cal year-year growth rate of total sales

In [28]:
query = """
WITH a AS
(
    SELECT 
        YEAR(orders.order_purchase_timestamp) AS years,
        ROUND(SUM(payments.payment_value),2) AS payments
    FROM orders
    JOIN payments
        ON orders.order_id = payments.order_id
    GROUP BY years
    ORDER BY years
)

SELECT 
    years,
    (
        (payments - LAG(payments,1) OVER(ORDER BY years))
        /
        LAG(payments,1) OVER(ORDER BY years)
    ) * 100 AS growth_percentage
FROM a
"""
cur.execute(query)
data = cur.fetchall()
data

df = pd.DataFrame(data,columns = ["years","yoy % growth"])
df




,years,yoy % growth
0,2016,NaN
1,2017,12112.703757
2,2018,20.000924


# retention rate in 6 months

In [31]:
query = """
with a as (select customers.customer_id,
min(orders.order_purchase_timestamp) first_order
from customers join orders
on customers.customer_id = orders.customer_id
group by  customers.customer_id),

b as (select a.customer_id,count(
distinct orders.order_purchase_timestamp) next_order
from a join orders
on orders.customer_id = a.customer_id 
and orders.order_purchase_timestamp > first_order
and orders.order_purchase_timestamp < 
date_add(first_order,interval 6 month)
group by a.customer_id)

select 100 * (count(distinct a.customer_id)/ count(distinct b.customer_id))
from a left join b
on a.customer_id = b.customer_id ;"""
cur.execute(query)
data = cur.fetchall()
data

[(None,)]

# identify top 3 cust who spent most money /year

In [32]:
query = """
select years, customer_id, payment, d_rank
from
(select year(orders.order_purchase_timestamp) years,
orders.customer_id,
sum(payments.payment_value) payment,
dense_rank() over (partition by year(orders.order_purchase_timestamp) 
order by sum(payments.payment_value) desc) d_rank
from orders join payments
on payments.order_id  = orders.order_id 
group by year(orders.order_purchase_timestamp),
orders.customer_id) as a
where d_rank <= 3 
"""

cur.execute(query)
data = cur.fetchall()
data

[(2016, 'a9dc96b027d1252bbac0a9b72d837fc6', 5694.2001953125, 1),
 (2016, '1d34ed25963d5aae4cf3d7f3a4cda173', 5602.9599609375, 2),
 (2016, '4a06381959b6670756de02e07b83815f', 4911.1201171875, 3),
 (2017, '1617b1357756262bfa56ab541c47bc16', 54656.3203125, 1),
 (2017, 'c6e2731c5b391845f6800c97401a43a9', 27717.240234375, 2),
 (2017, '3fd6777bbce08a352fddd04e4a7cc8f6', 26906.640625, 3),
 (2018, 'ec5b2ba62e574342386871631fafd3fc', 29099.51953125, 1),
 (2018, 'f48d464a0baaea338cb25f816991ab1f', 27688.83984375, 2),
 (2018, 'e0a2412720e9ea4f26c1ac985f6a7358', 19237.759765625, 3)]